In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

from sklearn.decomposition import PCA


# ============================================================
# 0. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 1. FFT preprocessing
#
# Original:
#   [1, 28, 28]
#
# FFT:
#   [1, 28, 28]
#
# spatial domain -> frequency domain
# ============================================================

def fft_transform(image):

    # image: [1, 28, 28]
    x = image.squeeze(0)

    # 2D FFT
    fft = torch.fft.fft2(x)

    # DC(low-frequency)를 중앙으로
    fft = torch.fft.fftshift(fft)

    # magnitude
    magnitude = torch.abs(fft)

    # dynamic range 압축
    magnitude = torch.log1p(magnitude)

    # normalize 0 ~ 1
    magnitude = (
        magnitude - magnitude.min()
    ) / (
        magnitude.max()
        - magnitude.min()
        + 1e-8
    )

    # [28, 28] -> [1, 28, 28]
    return magnitude.unsqueeze(0)


# ============================================================
# 2. MNIST
# ============================================================

transform = transforms.ToTensor()


train_raw = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_raw = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# ============================================================
# 3. FFT Dataset wrapper
# ============================================================

class FFTMNIST(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset


    def __len__(self):
        return len(self.dataset)


    def __getitem__(self, idx):

        image, label = self.dataset[idx]

        fft_image = fft_transform(image)

        return fft_image, label


train_dataset = FFTMNIST(
    train_raw
)

test_dataset = FFTMNIST(
    test_raw
)


train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)


# ============================================================
# 4. Sigmoid CNN
#
# FFT
# (B, 1, 28, 28)
#
#      ↓ Conv
#
# (B, 16, 28, 28)
#
#      ↓ Sigmoid
#      ↓ Pool
#
# (B, 16, 14, 14)
#
#      ↓ Conv
#
# (B, 32, 14, 14)
#
#      ↓ Sigmoid
#      ↓ Pool
#
# (B, 32, 7, 7)
#
#      ↓ Flatten
#
# (B, 1568)
#
#      ↓ FC + Sigmoid
#
# (B, 64)
#
#      ↓ FC
#
# (B, 10)
# ============================================================

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.sigmoid = nn.Sigmoid()

        self.fc1 = nn.Linear(
            32 * 7 * 7,
            64
        )

        self.fc2 = nn.Linear(
            64,
            10
        )


    def forward(
        self,
        x,
        return_features=False
    ):

        # ------------------------------------------
        # Block 1
        # ------------------------------------------

        conv1 = self.conv1(x)

        sigmoid1 = self.sigmoid(
            conv1
        )

        pool1 = self.pool(
            sigmoid1
        )


        # ------------------------------------------
        # Block 2
        # ------------------------------------------

        conv2 = self.conv2(
            pool1
        )

        sigmoid2 = self.sigmoid(
            conv2
        )

        pool2 = self.pool(
            sigmoid2
        )


        # ------------------------------------------
        # FC
        # ------------------------------------------

        flat = pool2.flatten(1)

        fc1 = self.fc1(
            flat
        )

        hidden = self.sigmoid(
            fc1
        )

        # CrossEntropyLoss 사용하므로
        # 마지막에는 sigmoid 안 씀
        logits = self.fc2(
            hidden
        )


        if return_features:

            return {
                "conv1": conv1,
                "sigmoid1": sigmoid1,
                "pool1": pool1,

                "conv2": conv2,
                "sigmoid2": sigmoid2,
                "pool2": pool2,

                "flat": flat,
                "fc1": fc1,
                "hidden": hidden,

                "logits": logits
            }


        return logits


model = CNN().to(device)


# ============================================================
# 5. Train
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


EPOCHS = 10


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    correct = 0
    total = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        optimizer.zero_grad()


        logits = model(
            images
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()

        optimizer.step()


        total_loss += loss.item()


        prediction = logits.argmax(
            dim=1
        )

        correct += (
            prediction == labels
        ).sum().item()

        total += labels.size(0)


    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} "
        f"| Loss: "
        f"{total_loss / len(train_loader):.4f} "
        f"| Train Acc: "
        f"{100 * correct / total:.2f}%"
    )


# ============================================================
# 6. Test
# ============================================================

model.eval()

correct = 0
total = 0


with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)


        logits = model(
            images
        )


        prediction = logits.argmax(
            dim=1
        )


        correct += (
            prediction == labels
        ).sum().item()

        total += labels.size(0)


accuracy = (
    correct / total
)


print()
print(
    f"Test Accuracy: "
    f"{accuracy * 100:.2f}%"
)


# ============================================================
# 7. Original vs FFT visualization
# ============================================================

N_IMAGES = 8


fig, axes = plt.subplots(
    2,
    N_IMAGES,
    figsize=(14, 4)
)


for i in range(N_IMAGES):

    image, label = test_raw[i]

    fft_image = fft_transform(
        image
    )


    # Original
    axes[0, i].imshow(
        image.squeeze(),
        cmap="gray"
    )

    axes[0, i].set_title(
        str(label)
    )

    axes[0, i].axis(
        "off"
    )


    # FFT
    axes[1, i].imshow(
        fft_image.squeeze(),
        cmap="gray"
    )

    axes[1, i].axis(
        "off"
    )


axes[0, 0].set_ylabel(
    "Original"
)

axes[1, 0].set_ylabel(
    "FFT"
)


plt.suptitle(
    "Spatial Domain vs Frequency Domain"
)

plt.tight_layout()

plt.show()


# ============================================================
# 8. One sample feature visualization
# ============================================================

index = 0


original_image, label = test_raw[
    index
]

fft_image = fft_transform(
    original_image
)


x = (
    fft_image
    .unsqueeze(0)
    .to(device)
)


model.eval()


with torch.no_grad():

    features = model(
        x,
        return_features=True
    )


prediction = (
    features["logits"]
    .argmax(dim=1)
    .item()
)


print()
print(
    "True label:",
    label
)

print(
    "Prediction:",
    prediction
)


# ============================================================
# 9. Feature map plotting
# ============================================================

def show_feature_maps(
    tensor,
    title,
    max_channels=16
):

    data = (
        tensor[0]
        .detach()
        .cpu()
        .numpy()
    )


    n = min(
        data.shape[0],
        max_channels
    )


    cols = 4

    rows = int(
        np.ceil(
            n / cols
        )
    )


    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            8,
            rows * 2
        )
    )


    axes = np.array(
        axes
    ).reshape(-1)


    for i in range(n):

        axes[i].imshow(
            data[i],
            cmap="gray"
        )

        axes[i].set_title(
            f"ch {i}"
        )

        axes[i].axis(
            "off"
        )


    for i in range(
        n,
        len(axes)
    ):

        axes[i].axis(
            "off"
        )


    plt.suptitle(
        f"{title}\nshape={data.shape}"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 10. FFT input
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6, 3)
)


axes[0].imshow(
    original_image.squeeze(),
    cmap="gray"
)

axes[0].set_title(
    "Original"
)

axes[0].axis(
    "off"
)


axes[1].imshow(
    fft_image.squeeze(),
    cmap="gray"
)

axes[1].set_title(
    "FFT input"
)

axes[1].axis(
    "off"
)


plt.tight_layout()
plt.show()


# ============================================================
# 11. CNN feature maps
# ============================================================

show_feature_maps(
    features["conv1"],
    "Conv1 BEFORE Sigmoid",
    16
)


show_feature_maps(
    features["sigmoid1"],
    "Conv1 AFTER Sigmoid",
    16
)


show_feature_maps(
    features["pool1"],
    "After Pool1",
    16
)


show_feature_maps(
    features["conv2"],
    "Conv2 BEFORE Sigmoid",
    16
)


show_feature_maps(
    features["sigmoid2"],
    "Conv2 AFTER Sigmoid",
    16
)


show_feature_maps(
    features["pool2"],
    "After Pool2",
    16
)


# ============================================================
# 12. Final hidden vector visualization
# ============================================================

hidden = (
    features["hidden"][0]
    .detach()
    .cpu()
    .numpy()
)


plt.figure(
    figsize=(12, 3)
)


plt.bar(
    np.arange(
        len(hidden)
    ),
    hidden
)


plt.xlabel(
    "Hidden dimension"
)

plt.ylabel(
    "Sigmoid activation"
)

plt.ylim(
    0,
    1
)

plt.title(
    "Final 64D Hidden Representation"
)

plt.grid(
    alpha=0.3
)

plt.show()


# ============================================================
# 13. CNN representations for PCA
# ============================================================

MAX_PCA_SAMPLES = 3000


fft_features_all = []

conv1_all = []
sigmoid1_all = []
pool1_all = []

conv2_all = []
sigmoid2_all = []
pool2_all = []

hidden_all = []

labels_all = []


model.eval()


count = 0


with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)


        features = model(
            images,
            return_features=True
        )


        # ------------------------------------------
        # input
        # ------------------------------------------

        fft_features_all.append(
            images
            .flatten(1)
            .cpu()
            .numpy()
        )


        # ------------------------------------------
        # CNN 1
        # ------------------------------------------

        conv1_all.append(
            features["conv1"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        sigmoid1_all.append(
            features["sigmoid1"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        pool1_all.append(
            features["pool1"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        # ------------------------------------------
        # CNN 2
        # ------------------------------------------

        conv2_all.append(
            features["conv2"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        sigmoid2_all.append(
            features["sigmoid2"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        pool2_all.append(
            features["pool2"]
            .flatten(1)
            .cpu()
            .numpy()
        )


        # ------------------------------------------
        # hidden
        # ------------------------------------------

        hidden_all.append(
            features["hidden"]
            .cpu()
            .numpy()
        )


        labels_all.append(
            labels.numpy()
        )


        count += len(labels)


        if count >= MAX_PCA_SAMPLES:
            break


# ============================================================
# 14. Concatenate
# ============================================================

fft_features_all = np.concatenate(
    fft_features_all
)[:MAX_PCA_SAMPLES]


conv1_all = np.concatenate(
    conv1_all
)[:MAX_PCA_SAMPLES]


sigmoid1_all = np.concatenate(
    sigmoid1_all
)[:MAX_PCA_SAMPLES]


pool1_all = np.concatenate(
    pool1_all
)[:MAX_PCA_SAMPLES]


conv2_all = np.concatenate(
    conv2_all
)[:MAX_PCA_SAMPLES]


sigmoid2_all = np.concatenate(
    sigmoid2_all
)[:MAX_PCA_SAMPLES]


pool2_all = np.concatenate(
    pool2_all
)[:MAX_PCA_SAMPLES]


hidden_all = np.concatenate(
    hidden_all
)[:MAX_PCA_SAMPLES]


labels_all = np.concatenate(
    labels_all
)[:MAX_PCA_SAMPLES]


# ============================================================
# 15. Shape 확인
# ============================================================

print()
print("============== FEATURE SHAPES ==============")

print(
    "FFT input :",
    fft_features_all.shape
)

print(
    "Conv1     :",
    conv1_all.shape
)

print(
    "Sigmoid1  :",
    sigmoid1_all.shape
)

print(
    "Pool1     :",
    pool1_all.shape
)

print(
    "Conv2     :",
    conv2_all.shape
)

print(
    "Sigmoid2  :",
    sigmoid2_all.shape
)

print(
    "Pool2     :",
    pool2_all.shape
)

print(
    "Hidden    :",
    hidden_all.shape
)


# ============================================================
# 16. PCA helper
# ============================================================

def pca_2d(data):

    pca = PCA(
        n_components=2
    )

    result = pca.fit_transform(
        data
    )

    print(
        "PCA variance:",
        pca.explained_variance_ratio_
    )

    return result


# ============================================================
# 17. PCA
# ============================================================

fft_2d = pca_2d(
    fft_features_all
)


conv1_2d = pca_2d(
    conv1_all
)


sigmoid1_2d = pca_2d(
    sigmoid1_all
)


pool1_2d = pca_2d(
    pool1_all
)


conv2_2d = pca_2d(
    conv2_all
)


sigmoid2_2d = pca_2d(
    sigmoid2_all
)


pool2_2d = pca_2d(
    pool2_all
)


hidden_2d = pca_2d(
    hidden_all
)


# ============================================================
# 18. PCA visualization
# ============================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(22, 11)
)


representations = [

    (
        fft_2d,
        "FFT Input"
    ),

    (
        conv1_2d,
        "Conv1"
    ),

    (
        sigmoid1_2d,
        "After Sigmoid1"
    ),

    (
        pool1_2d,
        "Pool1"
    ),

    (
        conv2_2d,
        "Conv2"
    ),

    (
        sigmoid2_2d,
        "After Sigmoid2"
    ),

    (
        pool2_2d,
        "Pool2"
    ),

    (
        hidden_2d,
        "64D Hidden"
    )
]


for ax, (
    data,
    title
) in zip(
    axes.flat,
    representations
):

    scatter = ax.scatter(
        data[:, 0],
        data[:, 1],

        c=labels_all,

        cmap="tab10",

        s=7,

        alpha=0.55
    )


    ax.set_title(
        title
    )

    ax.set_xlabel(
        "PC1"
    )

    ax.set_ylabel(
        "PC2"
    )

    ax.grid(alpha=0.2)


plt.colorbar(
    scatter,
    ax=axes.ravel().tolist(),
    label="MNIST digit"
)


plt.suptitle(
    "Representation Space Through FFT + Sigmoid CNN",
    fontsize=16
)

plt.show()

/home/br4c3/Projects/space-war/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Device: cpu
Epoch 01/10 | Loss: 2.3053 | Train Acc: 10.69%
